# Assignment: PyTorch Fundamentals
## Tensors, Autograd & Neural Network Training | California Housing

---

### Assignment Objectives

By completing this assignment you will:

1. **Create and manipulate** PyTorch tensors: shapes, dtypes, indexing, reshaping, and device management.
2. **Use autograd** to compute gradients automatically and verify them against analytical derivatives.
3. **Build** a neural network using `nn.Module` with custom architecture for regression.
4. **Implement** the complete PyTorch training loop: forward pass, loss, backward pass, optimizer step.
5. **Train** a network on the California Housing dataset using `DataLoader` and evaluate with standard metrics.
6. **Experiment** with learning rates, architectures, and optimizers, documenting your findings.

> **Deliverable:** After running every cell, you will have all the figures, tables, and analysis needed to write a professional lab report.

---

### Report Requirements

Throughout this notebook you will see **“For your report”** prompts. These guide what to write in your lab report. Include all generated figures and answer every prompt.

---

# Part 1: Background — PyTorch & Tensors

## 1.1 What is PyTorch?

**PyTorch** is an open-source deep learning framework developed by Meta AI. It provides two core features:

1. **Tensors** — multi-dimensional arrays (like NumPy) that can run on GPUs
2. **Autograd** — automatic differentiation for computing gradients

Together, these make it possible to define, train, and evaluate neural networks with clean, Pythonic code.

## 1.2 Tensors

A **tensor** is a generalization of scalars, vectors, and matrices to arbitrary dimensions:

```
Scalar  (0D):  42              → torch.tensor(42.0)
Vector  (1D):  [1, 2, 3]      → torch.tensor([1, 2, 3])
Matrix  (2D):  [[1,2],[3,4]]  → torch.tensor([[1,2],[3,4]])
3D Tensor:     batch of matrices → torch.rand(2, 3, 4)
```

Key properties: **shape**, **dtype**, **device**, **requires_grad**.

## 1.3 Autograd & The Training Loop

PyTorch tracks operations on tensors and automatically computes gradients via **backpropagation**. The core training pattern:

```python
predictions = model(X_batch)       # 1. Forward pass
loss = criterion(predictions, y)   # 2. Compute loss
loss.backward()                    # 3. Backward pass (compute gradients)
optimizer.step()                   # 4. Update weights
optimizer.zero_grad()              # 5. Reset gradients
```

## 1.4 nn.Module

All PyTorch models inherit from `nn.Module`. A model defines:
- `__init__()`: declare layers (learnable parameters)
- `forward(x)`: define how data flows through the layers

```python
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(8, 64)
        self.relu = nn.ReLU()
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        return self.output(self.relu(self.layer1(x)))
```

---

# Part 2: Environment Setup

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  INSTALL DEPENDENCIES (run once)                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
%pip install torch numpy matplotlib pandas scikit-learn --quiet

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  IMPORTS                                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ── Plotting defaults ────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

# ── Device selection ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

# ── Reproducibility ──────────────────────────────────────────────────────────────
np.random.seed(42)
torch.manual_seed(42)

---

# Part 3: Tensor Operations

## 3.1 Creating Tensors

Tensors are the fundamental data structure in PyTorch. In this section you will create tensors of different dimensionalities and explore their properties.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CREATE TENSORS OF EVERY DIMENSIONALITY                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
torch.manual_seed(42)

# Create tensors of different dimensionalities
scalar = torch.tensor(42.0)                # 0D — a single number
vector = torch.tensor([1.0, 2.0, 3.0])    # 1D — a row of numbers
matrix = torch.tensor([[1, 2, 3],          # 2D — rows and columns
                        [4, 5, 6]])
tensor_3d = torch.rand(2, 3, 4)           # 3D — e.g., batch of matrices

for name, t in [('Scalar', scalar), ('Vector', vector),
                ('Matrix', matrix), ('3D Tensor', tensor_3d)]:
    print(f'{name:12s} | shape: {str(t.shape):15s} | ndim: {t.ndim} | '
          f'numel: {t.numel():4d} | dtype: {t.dtype}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TENSOR CREATION METHODS                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
torch.manual_seed(42)

print('zeros(2,3)     :', torch.zeros(2, 3))
print('ones(2,3)      :', torch.ones(2, 3))
print('rand(2,3)      :', torch.rand(2, 3))         # Uniform [0,1)
print('randn(2,3)     :', torch.randn(2, 3))        # Normal(0,1)
print('arange(0,10,2) :', torch.arange(0, 10, 2))   # Like Python range
print('linspace(0,1,5):', torch.linspace(0, 1, 5))  # Evenly spaced
print('full(2,3,val=7):', torch.full((2, 3), 7.0))
print('eye(3)         :', torch.eye(3))              # Identity matrix

## 3.2 Element-wise & Matrix Operations

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ELEMENT-WISE vs MATRIX OPERATIONS                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
torch.manual_seed(42)

a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

print('--- Element-wise Operations ---')
print(f'a + b  = {a + b}')            # Addition
print(f'a * b  = {a * b}')            # Element-wise multiply (Hadamard)
print(f'a ** 2 = {a ** 2}')           # Element-wise power

print('\n--- Matrix Multiplication ---')
print(f'a @ b       = {a @ b}')        # @ operator
print(f'torch.matmul= {torch.matmul(a, b)}')

print('\n--- Aggregation ---')
print(f'a.sum()     = {a.sum()}')
print(f'a.mean()    = {a.mean()}')
print(f'a.max()     = {a.max()}')
print(f'a.sum(dim=0)= {a.sum(dim=0)}  (sum down columns)')
print(f'a.sum(dim=1)= {a.sum(dim=1)}  (sum across rows)')

## 3.3 Indexing, Slicing & Reshaping

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  INDEXING, SLICING & RESHAPING                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
torch.manual_seed(42)

# --- Indexing & Slicing ---
x = torch.arange(1, 13).reshape(3, 4).float()
print(f'x = \n{x}\n')
print(f'x[0]        = {x[0]}        -- first row')
print(f'x[:, 1]     = {x[:, 1]}     -- second column')
print(f'x[0:2, 1:3] = \n{x[0:2, 1:3]}  -- sub-matrix')
print(f'x[-1]       = {x[-1]}       -- last row')
print(f'x[1, 2]     = {x[1, 2]}            -- single element')

# --- Reshaping ---
print('\n--- Reshaping ---')
original = torch.arange(24).float()
print(f'Original: shape={original.shape}')

reshaped = original.view(4, 6)
print(f'view(4,6):    shape={reshaped.shape}')

reshaped_3d = original.reshape(2, 3, 4)
print(f'reshape(2,3,4): shape={reshaped_3d.shape}')

flat = reshaped_3d.flatten()
print(f'flatten():    shape={flat.shape}')

# squeeze / unsqueeze
t = torch.zeros(1, 3, 1)
print(f'\nsqueeze:   {t.shape} -> {t.squeeze().shape}')
t2 = torch.zeros(3)
print(f'unsqueeze: {t2.shape} -> {t2.unsqueeze(0).shape} (add batch dim)')

## 3.4 NumPy Interop, Devices & Data Types

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  NUMPY INTEROP, DEVICES & DTYPES                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# NumPy <-> Tensor (shared memory!)
np_array = np.array([1.0, 2.0, 3.0])
tensor_from_np = torch.from_numpy(np_array)
print(f'NumPy array  : {np_array}')
print(f'Torch tensor : {tensor_from_np}')

np_array[0] = 999
print(f'\nAfter modifying NumPy array[0] = 999:')
print(f'NumPy array  : {np_array}')
print(f'Torch tensor : {tensor_from_np}  <-- also changed (shared memory!)')

# Device management
print(f'\nDefault device: {torch.tensor([1.0]).device}')
t = torch.randn(3, 3).to(device)
print(f'Tensor on {device}: {t.device}')

# Data types
print('\n--- Data Types ---')
t32 = torch.tensor([1.0, 2.0, 3.0])                    # float32 (default)
t64 = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
t_int = torch.tensor([1, 2, 3])                         # int64 (default)
print(f'float32: {t32.dtype} | bytes/element: {t32.element_size()}')
print(f'float64: {t64.dtype} | bytes/element: {t64.element_size()}')
print(f'int64  : {t_int.dtype} | bytes/element: {t_int.element_size()}')
print(f'\nCast: t32.to(torch.float64).dtype = {t32.to(torch.float64).dtype}')

> **For your report:** Explain what a tensor is and how it generalizes scalars, vectors, and matrices. Why does PyTorch default to `float32` instead of `float64`? What does "shared memory" between NumPy and PyTorch mean, and when could it cause bugs?

---

# Part 4: Autograd — Automatic Differentiation

## 4.1 How Autograd Works

When a tensor has `requires_grad=True`, PyTorch records every operation on it in a **computational graph**. Calling `.backward()` traverses this graph in reverse to compute gradients via the **chain rule**.

```
x (requires_grad=True)
 │
 │  y = x²
 │
 ▼
y.backward()  →  x.grad = dy/dx = 2x
```

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SIMPLE AUTOGRAD: y = x²                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2          # y = 9
y.backward()        # compute dy/dx

print(f'x     = {x.item()}')
print(f'y=x²  = {y.item()}')
print(f'dy/dx = {x.grad.item()}  (should be 2*3 = 6)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  COMPUTATIONAL GRAPH & CHAIN RULE                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Linear function: y = w*x + b
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

y = w * x + b  # y = 3*2 + 1 = 7
y.backward()

print(f'y = w*x + b = {w.item()}*{x.item()} + {b.item()} = {y.item()}')
print(f'dy/dx = {x.grad.item()} (should be w = {w.item()})')
print(f'dy/dw = {w.grad.item()} (should be x = {x.item()})')
print(f'dy/db = {b.grad.item()} (should be 1)')

# Chain rule: y = ((x + 2)²) * 3
# dy/dx = 3 * 2*(x+2) = 6*(x+2)
print('\n--- Chain Rule Demo ---')
x2 = torch.tensor(1.0, requires_grad=True)
y2 = ((x2 + 2) ** 2) * 3
y2.backward()

print(f'y = ((x+2)²) * 3')
print(f'At x = {x2.item()}: y = {y2.item()}')
print(f'PyTorch gradient: {x2.grad.item()}')
print(f'Analytical:       6*(x+2) = 6*(1+2) = 18.0')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  no_grad() AND detach()                                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

x = torch.tensor(3.0, requires_grad=True)

# torch.no_grad() — stops tracking for inference (saves memory)
with torch.no_grad():
    y_no_track = x * 2
    print(f'no_grad: requires_grad = {y_no_track.requires_grad}')

# detach() — extract a tensor from the graph
y = x ** 2
y_detached = y.detach()
print(f'detach:  requires_grad = {y_detached.requires_grad}')
print(f'\nUse no_grad() during inference to save memory.')
print(f'Use detach() to extract values for plotting/logging.')

## 4.2 Autograd Visualizer

Visualize how PyTorch computes derivatives for multiple functions.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  AUTOGRAD VISUALIZER                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

functions = {
    'y = x²':     (lambda x: x**2,      lambda x: 2*x),
    'y = sin(x)':  (lambda x: torch.sin(x), lambda x: torch.cos(x)),
    'y = e^x':     (lambda x: torch.exp(x), lambda x: torch.exp(x)),
    'y = x³ - 2x': (lambda x: x**3 - 2*x, lambda x: 3*x**2 - 2),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
x_range = np.linspace(-3, 3, 200)

for ax, (name, (fn, grad_fn)) in zip(axes.flat, functions.items()):
    # Compute function values
    x_t = torch.tensor(x_range, dtype=torch.float32)
    y_vals = fn(x_t).detach().numpy()

    # Compute gradients at each point via autograd
    grads = []
    for xi in x_range:
        xi_t = torch.tensor(xi, requires_grad=True)
        yi = fn(xi_t)
        yi.backward()
        grads.append(xi_t.grad.item())

    ax.plot(x_range, y_vals, 'b-', linewidth=2, label=name)
    ax.plot(x_range, grads, 'r--', linewidth=2, label=f'd/dx ({name})')
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.set_title(name, fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('asgn_fig_autograd_functions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_autograd_functions.png')

> **For your report:** Explain how autograd works. What is the computational graph? Why is automatic differentiation essential for training neural networks? For each function plotted, verify that the gradient matches the analytical derivative.

---

# Part 5: California Housing Dataset

## 5.1 About the Dataset

The **California Housing** dataset contains 20,640 samples of California census block groups from the 1990 census. Each sample has **8 numerical features** and the target is the **median house value** (in $100,000s).

| Feature | Description |
|---|---|
| MedInc | Median income in block group |
| HouseAge | Median house age in block group |
| AveRooms | Average number of rooms per household |
| AveBedrms | Average number of bedrooms per household |
| Population | Block group population |
| AveOccup | Average number of household members |
| Latitude | Block group latitude |
| Longitude | Block group longitude |

**Target**: Median house value (in $100,000s), ranging from 0.15 to 5.0

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LOAD CALIFORNIA HOUSING DATASET                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

housing = fetch_california_housing()
X_raw = housing.data          # (20640, 8)
y_raw = housing.target        # (20640,)
feature_names = housing.feature_names

df = pd.DataFrame(X_raw, columns=feature_names)
df["MedHouseVal"] = y_raw

print(f"Dataset shape: {X_raw.shape}")
print(f"Target shape:  {y_raw.shape}")
print(f"Features: {feature_names}")
print()
df.head(10)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STATISTICAL SUMMARY                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print(df.describe().round(3))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  VISUALIZE FEATURE DISTRIBUTIONS                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
all_cols = list(feature_names) + ["MedHouseVal"]

for i, (ax, col) in enumerate(zip(axes.flat, all_cols)):
    ax.hist(df[col], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    ax.set_title(col, fontsize=11)
    ax.set_ylabel('Count')
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature Distributions — California Housing', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('asgn_fig_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_feature_distributions.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CORRELATION WITH TARGET                                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, (ax, name) in enumerate(zip(axes.flat, feature_names)):
    ax.scatter(df[name], df['MedHouseVal'], alpha=0.1, s=5, color='steelblue')
    corr = df[name].corr(df['MedHouseVal'])
    ax.set_title(f'{name}\nr = {corr:.3f}', fontsize=10)
    ax.set_xlabel(name)
    ax.set_ylabel('MedHouseVal')
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature vs Target Correlations', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('asgn_fig_correlations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_correlations.png')

## 5.2 Preprocessing & Conversion to Tensors

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  NORMALIZE, SPLIT & CONVERT TO TENSORS                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Train/test split (80/20)
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_scaled, y_raw, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1)  # (N,) -> (N,1)
X_test  = torch.tensor(X_test_np,  dtype=torch.float32)
y_test  = torch.tensor(y_test_np,  dtype=torch.float32).unsqueeze(1)

print(f'X_train: {X_train.shape}  dtype={X_train.dtype}')
print(f'y_train: {y_train.shape}  dtype={y_train.dtype}')
print(f'X_test:  {X_test.shape}')
print(f'y_test:  {y_test.shape}')
print(f'\nNote: .unsqueeze(1) converts (N,) -> (N,1) so shapes match model output')

> **For your report:** Describe the California Housing dataset. How many samples and features does it have? Why do we normalize the features before training? Why do we need `.unsqueeze(1)` on the target?

---

# Part 6: Building a Neural Network with nn.Module

## 6.1 nn.Linear — The Fundamental Layer

A `nn.Linear(in, out)` layer computes `y = xWᵀ + b` where W has shape `(out, in)` and b has shape `(out,)`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  nn.Linear EXPLAINED                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
torch.manual_seed(42)

# A linear layer: 8 inputs -> 4 outputs
layer = nn.Linear(8, 4)

print(f'Layer: {layer}')
print(f'Weight shape: {layer.weight.shape}  (out_features x in_features)')
print(f'Bias shape  : {layer.bias.shape}    (out_features)')
print(f'\nWeight matrix:\n{layer.weight.data}')
print(f'\nBias vector: {layer.bias.data}')

# Forward pass
x = torch.randn(1, 8)  # batch_size=1, features=8
output = layer(x)
print(f'\nInput shape:  {x.shape}')
print(f'Output shape: {output.shape}')
print(f'Output:       {output.data}')

## 6.2 Building Models: Sequential vs Custom nn.Module

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  nn.Sequential MODEL                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
torch.manual_seed(42)

# Quick way: nn.Sequential
model_seq = nn.Sequential(
    nn.Linear(8, 64),   # 8 input features -> 64 hidden neurons
    nn.ReLU(),           # Activation
    nn.Linear(64, 32),  # 64 -> 32
    nn.ReLU(),
    nn.Linear(32, 1),   # 32 -> 1 output (price)
)

print('Sequential model:')
print(model_seq)

# Test
x_test_batch = torch.randn(5, 8)
preds = model_seq(x_test_batch)
print(f'\nInput:  {x_test_batch.shape}')
print(f'Output: {preds.shape}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CUSTOM nn.Module CLASS                                                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
torch.manual_seed(42)

class HousingNet(nn.Module):
    """Neural network for California Housing price prediction."""

    def __init__(self, input_dim=8, hidden1=64, hidden2=32):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden1)
        self.layer2 = nn.Linear(hidden1, hidden2)
        self.output_layer = nn.Linear(hidden2, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.output_layer(x)
        return x

model = HousingNet()
print('Custom HousingNet:')
print(model)

# Test forward pass
x_dummy = torch.randn(5, 8)
out = model(x_dummy)
print(f'\nInput:  {x_dummy.shape}')
print(f'Output: {out.shape}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  COUNT PARAMETERS                                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print(f'{"Layer":<25} {"Shape":<20} {"# Params":<10}')
print('-' * 55)
total = 0
for name, param in model.named_parameters():
    n = param.numel()
    total += n
    print(f'{name:<25} {str(tuple(param.shape)):<20} {n:<10}')
print('-' * 55)
print(f'{"TOTAL":<25} {"":<20} {total:<10}')
print(f'\nAll these numbers are what the network LEARNS during training!')

> **For your report:** Explain `nn.Module` and `nn.Linear`. What is the difference between `nn.Sequential` and a custom `nn.Module` class? How many total parameters does the [8, 64, 32, 1] network have? Show the math for at least one layer.

---

# Part 7: Loss Functions & Optimizers

## 7.1 Loss Functions

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LOSS FUNCTIONS                                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# --- MSELoss for regression ---
criterion = nn.MSELoss()

predictions = torch.tensor([2.5, 3.0, 4.5])
actual      = torch.tensor([2.0, 3.5, 4.0])

# Manual MSE
errors = predictions - actual
squared = errors ** 2
manual_mse = squared.mean()

# PyTorch MSE
pytorch_mse = criterion(predictions, actual)

print('MSELoss (Regression):')
print(f'  Predictions : {predictions.tolist()}')
print(f'  Actual      : {actual.tolist()}')
print(f'  Errors      : {errors.tolist()}')
print(f'  Squared     : {squared.tolist()}')
print(f'  Manual MSE  : {manual_mse.item():.4f}')
print(f'  PyTorch MSE : {pytorch_mse.item():.4f}')

# --- CrossEntropyLoss for classification ---
print('\nCrossEntropyLoss (Classification):')
ce_loss = nn.CrossEntropyLoss()
logits = torch.tensor([[2.0, 1.0, 0.1],    # sample 1: class 0 most likely
                        [0.5, 2.5, 0.3]])   # sample 2: class 1 most likely
labels = torch.tensor([0, 1])               # true labels
loss_ce = ce_loss(logits, labels)
print(f'  Cross-Entropy Loss: {loss_ce.item():.4f}')

# Quick reference
print('\n--- Choosing the Right Loss ---')
print(f'{"Task":<25} {"Loss Function":<25}')
print('-' * 50)
print(f'{"Regression":<25} {"nn.MSELoss()":<25}')
print(f'{"Binary Classification":<25} {"nn.BCEWithLogitsLoss()":<25}')
print(f'{"Multi-class":<25} {"nn.CrossEntropyLoss()":<25}')

## 7.2 Optimizers

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  OPTIMIZERS                                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
torch.manual_seed(42)
model_for_opt = HousingNet()

# SGD
optimizer_sgd = optim.SGD(model_for_opt.parameters(), lr=0.01)
print('SGD optimizer:')
print(f'  Learning rate: {optimizer_sgd.param_groups[0]["lr"]}')

# Adam
optimizer_adam = optim.Adam(model_for_opt.parameters(), lr=0.001)
print(f'\nAdam optimizer:')
print(f'  Learning rate: {optimizer_adam.param_groups[0]["lr"]}')
print(f'  Betas: {optimizer_adam.param_groups[0]["betas"]}')

# The critical 4-line pattern
print('\n' + '=' * 50)
print('THE 4-LINE TRAINING PATTERN (every batch):')
print('=' * 50)
print('1. predictions = model(X_batch)       # Forward pass')
print('2. loss = criterion(predictions, y)    # Compute loss')
print('3. loss.backward()                     # Backward pass')
print('4. optimizer.step()                    # Update weights')
print('   optimizer.zero_grad()               # Reset gradients')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LEARNING RATE VISUALIZATION                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Minimize L(w) = (w - 3)² with different learning rates
learning_rates = [0.01, 0.1, 0.5, 0.95]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, lr in zip(axes.flat, learning_rates):
    w = 0.0  # start far from minimum
    w_history = [w]
    for _ in range(30):
        grad = 2 * (w - 3)  # dL/dw = 2*(w-3)
        w = w - lr * grad
        w_history.append(w)

    w_plot = np.linspace(-1, 7, 200)
    loss_plot = (w_plot - 3) ** 2
    ax.plot(w_plot, loss_plot, 'b-', linewidth=2)
    ax.plot(w_history, [(w_i - 3)**2 for w_i in w_history], 'ro-', markersize=4)
    ax.set_title(f'lr = {lr}', fontsize=12)
    ax.set_xlabel('w')
    ax.set_ylabel('Loss')
    ax.grid(True, alpha=0.3)

plt.suptitle('Effect of Learning Rate on Gradient Descent', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('asgn_fig_learning_rate_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_learning_rate_demo.png')

> **For your report:** Why is MSE the right loss for this regression task? Explain the 4-line training pattern and why each step is necessary. What happens when the learning rate is too small, too large, or just right? Include the learning rate figure.

---

# Part 8: Dataset & DataLoader

`TensorDataset` wraps tensors into a dataset. `DataLoader` handles batching, shuffling, and iteration.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CREATE TENSORDATASET & DATALOADER                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

BATCH_SIZE = 64

train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Batch size    : {BATCH_SIZE}')
print(f'Train samples : {len(train_dataset)}')
print(f'Train batches : {len(train_loader)}')
print(f'Test samples  : {len(test_dataset)}')
print(f'Test batches  : {len(test_loader)}')

# Peek at first batch
X_batch, y_batch = next(iter(train_loader))
print(f'\nFirst batch:')
print(f'  X_batch shape: {X_batch.shape}')
print(f'  y_batch shape: {y_batch.shape}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ITERATE OVER DATALOADER                                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print('Iterating over DataLoader (first 5 batches):')
for i, (X_b, y_b) in enumerate(train_loader):
    if i >= 5:
        break
    print(f'  Batch {i}: X={X_b.shape}, y={y_b.shape}, '
          f'y_mean={y_b.mean():.3f}')

> **For your report:** What is the purpose of `DataLoader`? Why do we use mini-batches instead of the full dataset? Why do we shuffle the training data but not the test data?

---

# Part 9: The Training Loop

## 9.1 Step-by-Step: One Training Iteration

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  ONE TRAINING STEP (DETAILED)                                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
torch.manual_seed(42)

step_model = HousingNet()
step_optimizer = optim.Adam(step_model.parameters(), lr=0.001)
step_criterion = nn.MSELoss()

# Get one batch
X_batch, y_batch = next(iter(train_loader))

print('=== One Training Step (Detailed) ===')
print()

# Step 1: Forward pass
predictions = step_model(X_batch)
print(f'Step 1 - Forward pass:')
print(f'  Input:  {X_batch.shape}')
print(f'  Output: {predictions.shape}')

# Step 2: Compute loss
loss = step_criterion(predictions, y_batch)
print(f'\nStep 2 - Compute loss:')
print(f'  MSE Loss: {loss.item():.4f}')

# Step 3: Backward pass
loss.backward()
print(f'\nStep 3 - Backward pass:')
print(f'  Gradients computed for {sum(1 for _ in step_model.parameters())} parameter tensors')
for name, param in step_model.named_parameters():
    if param.grad is not None:
        print(f'  {name}: grad norm = {param.grad.norm():.4f}')

# Step 4: Update weights
w_before = step_model.layer1.weight.data[0, 0].item()
step_optimizer.step()
w_after = step_model.layer1.weight.data[0, 0].item()
print(f'\nStep 4 - Optimizer step:')
print(f'  layer1.weight[0,0]: {w_before:.6f} -> {w_after:.6f}')

# Step 5: Zero gradients
step_optimizer.zero_grad()
print(f'\nStep 5 - Zero gradients:')
print(f'  layer1.weight.grad norm: {step_model.layer1.weight.grad.norm():.4f} (should be 0)')

## 9.2 Full Training Loop

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  FULL TRAINING LOOP                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
torch.manual_seed(42)

# Create model, loss, optimizer
model = HousingNet(input_dim=8, hidden1=64, hidden2=32)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Move to device
model = model.to(device)

# Training settings
NUM_EPOCHS = 100
train_losses = []
test_losses = []

print(f'Training HousingNet for {NUM_EPOCHS} epochs...')
print(f'Device: {device}')
print('-' * 60)

for epoch in range(NUM_EPOCHS):
    # --- Training ---
    model.train()
    epoch_loss = 0.0
    n_batches = 0

    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)

        # The 4-line pattern
        predictions = model(X_b)
        loss = criterion(predictions, y_b)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        n_batches += 1

    avg_train_loss = epoch_loss / n_batches
    train_losses.append(avg_train_loss)

    # --- Evaluation ---
    model.eval()
    with torch.no_grad():
        test_pred = model(X_test.to(device))
        test_loss = criterion(test_pred, y_test.to(device)).item()
    test_losses.append(test_loss)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS} | '
              f'Train Loss: {avg_train_loss:.4f} | '
              f'Test Loss: {test_loss:.4f}')

print('-' * 60)
print(f'Final Train Loss: {train_losses[-1]:.4f}')
print(f'Final Test Loss:  {test_losses[-1]:.4f}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PLOT TRAINING CURVES                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Full range
ax1.plot(train_losses, label='Train Loss', color='steelblue', linewidth=2)
ax1.plot(test_losses, label='Test Loss', color='darkorange', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.set_title('Training Curves')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Last 50 epochs (zoomed)
ax2.plot(range(50, NUM_EPOCHS), train_losses[50:], label='Train Loss',
         color='steelblue', linewidth=2)
ax2.plot(range(50, NUM_EPOCHS), test_losses[50:], label='Test Loss',
         color='darkorange', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE Loss')
ax2.set_title('Training Curves (Last 50 Epochs)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('asgn_fig_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_training_curves.png')

> **For your report:** Include the training curves figure. Is the model converging? Is there a gap between training and test loss (overfitting)? At what epoch does the loss roughly plateau? Explain the difference between `model.train()` and `model.eval()`.

---

# Part 10: Evaluation & Visualization

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TEST SET EVALUATION                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

model.eval()
with torch.no_grad():
    test_predictions = model(X_test.to(device)).cpu()

# Metrics
mse = ((test_predictions - y_test) ** 2).mean().item()
rmse = mse ** 0.5
mae = (test_predictions - y_test).abs().mean().item()
ss_res = ((y_test - test_predictions) ** 2).sum().item()
ss_tot = ((y_test - y_test.mean()) ** 2).sum().item()
r2 = 1 - ss_res / ss_tot

print('Test Set Evaluation')
print('=' * 40)
print(f'MSE  : {mse:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R\u00b2   : {r2:.4f}')
print(f'\nR\u00b2 = {r2:.4f} means the model explains {r2*100:.1f}% of price variance.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PREDICTED VS ACTUAL + RESIDUALS                                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Predicted vs Actual
ax1.scatter(y_test.numpy(), test_predictions.numpy(), alpha=0.3, s=10, color='steelblue')
lims = [min(y_test.min().item(), test_predictions.min().item()),
        max(y_test.max().item(), test_predictions.max().item())]
ax1.plot(lims, lims, 'r--', linewidth=2, label='Perfect (y=x)')
ax1.set_xlabel('Actual Price ($100K)', fontsize=12)
ax1.set_ylabel('Predicted Price ($100K)', fontsize=12)
ax1.set_title(f'Predicted vs Actual (R\u00b2 = {r2:.4f})', fontsize=13)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Residuals
residuals = (test_predictions - y_test).numpy().flatten()
ax2.hist(residuals, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Prediction Error ($100K)')
ax2.set_ylabel('Count')
ax2.set_title('Residual Distribution')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('asgn_fig_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_evaluation.png')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  BEST AND WORST PREDICTIONS                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

errors = (test_predictions - y_test).abs().flatten()
best_idx = errors.argsort()[:5]
worst_idx = errors.argsort()[-5:]

print('Best 5 Predictions:')
print(f'{"Actual":>10} {"Predicted":>10} {"Error":>10}')
print('-' * 32)
for idx in best_idx:
    print(f'{y_test[idx].item():10.4f} {test_predictions[idx].item():10.4f} '
          f'{errors[idx].item():10.4f}')

print(f'\nWorst 5 Predictions:')
print(f'{"Actual":>10} {"Predicted":>10} {"Error":>10}')
print('-' * 32)
for idx in worst_idx:
    print(f'{y_test[idx].item():10.4f} {test_predictions[idx].item():10.4f} '
          f'{errors[idx].item():10.4f}')

> **For your report:** Report all metrics (MSE, RMSE, MAE, R\u00b2). Include the predicted vs actual scatter plot and residual histogram. Are the residuals normally distributed? What patterns do you see in the best and worst predictions?

---

# Part 11: Hyperparameter Experiments

## 11.1 Helper Function

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  QUICK TRAIN HELPER                                                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def quick_train(model, train_loader, test_X, test_y, lr=0.001,
                epochs=80, optimizer_cls=optim.Adam):
    """Train a model and return loss history."""
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optimizer_cls(model.parameters(), lr=lr)
    train_hist, test_hist = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        n = 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            pred = model(X_b)
            loss = criterion(pred, y_b)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            epoch_loss += loss.item()
            n += 1
        train_hist.append(epoch_loss / n)

        model.eval()
        with torch.no_grad():
            t_pred = model(test_X.to(device))
            t_loss = criterion(t_pred, test_y.to(device)).item()
        test_hist.append(t_loss)

    # Final R²
    model.eval()
    with torch.no_grad():
        final_pred = model(test_X.to(device)).cpu()
    ss_res = ((test_y - final_pred) ** 2).sum().item()
    ss_tot = ((test_y - test_y.mean()) ** 2).sum().item()
    r2 = 1 - ss_res / ss_tot

    return train_hist, test_hist, r2

## 11.2 Experiment A: Architecture Comparison

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT A: ARCHITECTURE                                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

architectures = {
    'Small [8,16,1]':    (16, 0),     # 1 hidden layer
    'Medium [8,64,32,1]': (64, 32),   # 2 hidden layers
    'Large [8,128,64,1]': (128, 64),  # 2 hidden layers, wider
}

fig, ax = plt.subplots(figsize=(10, 6))
results_arch = {}

for name, (h1, h2) in architectures.items():
    torch.manual_seed(42)
    if h2 == 0:
        m = nn.Sequential(nn.Linear(8, h1), nn.ReLU(), nn.Linear(h1, 1))
    else:
        m = HousingNet(input_dim=8, hidden1=h1, hidden2=h2)
    train_h, test_h, r2_val = quick_train(m, train_loader, X_test, y_test)
    ax.plot(test_h, label=f'{name} (R\u00b2={r2_val:.4f})', linewidth=2)
    results_arch[name] = r2_val
    print(f'{name}: R\u00b2 = {r2_val:.4f}')

ax.set_xlabel('Epoch')
ax.set_ylabel('Test Loss (MSE)')
ax.set_title('Experiment A: Architecture Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_exp_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_exp_architecture.png')

## 11.3 Experiment B: Learning Rate

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT B: LEARNING RATE                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

learning_rates_exp = [0.0001, 0.001, 0.01, 0.05]

fig, ax = plt.subplots(figsize=(10, 6))
results_lr = {}

for lr in learning_rates_exp:
    torch.manual_seed(42)
    m = HousingNet()
    train_h, test_h, r2_val = quick_train(m, train_loader, X_test, y_test, lr=lr)
    ax.plot(test_h, label=f'lr={lr} (R\u00b2={r2_val:.4f})', linewidth=2)
    results_lr[lr] = r2_val
    print(f'lr={lr}: R\u00b2 = {r2_val:.4f}')

ax.set_xlabel('Epoch')
ax.set_ylabel('Test Loss (MSE)')
ax.set_title('Experiment B: Learning Rate Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_exp_learning_rate.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_exp_learning_rate.png')

## 11.4 Experiment C: Optimizer Comparison

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  EXPERIMENT C: OPTIMIZER COMPARISON                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

optimizers_exp = {
    'SGD (lr=0.01)':  (optim.SGD, 0.01),
    'SGD+Momentum':   (lambda p, lr: optim.SGD(p, lr=lr, momentum=0.9), 0.01),
    'Adam (lr=0.001)': (optim.Adam, 0.001),
    'AdamW (lr=0.001)': (optim.AdamW, 0.001),
}

fig, ax = plt.subplots(figsize=(10, 6))
results_opt = {}

for name, (opt_cls, lr) in optimizers_exp.items():
    torch.manual_seed(42)
    m = HousingNet()
    train_h, test_h, r2_val = quick_train(
        m, train_loader, X_test, y_test, lr=lr, optimizer_cls=opt_cls
    )
    ax.plot(test_h, label=f'{name} (R\u00b2={r2_val:.4f})', linewidth=2)
    results_opt[name] = r2_val
    print(f'{name}: R\u00b2 = {r2_val:.4f}')

ax.set_xlabel('Epoch')
ax.set_ylabel('Test Loss (MSE)')
ax.set_title('Experiment C: Optimizer Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('asgn_fig_exp_optimizer.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: asgn_fig_exp_optimizer.png')

> **For your report:** For each experiment, include the comparison plot and state which setting performed best. Explain WHY: consider model capacity, convergence speed, and how each optimizer handles gradients. Recommend the best setting for each hyperparameter.

---

# Part 12: Common Mistakes & Best Practices

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  COMMON MISTAKES DEMO                                                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Mistake #1: Forgetting zero_grad()
print('MISTAKE #1: Forgetting zero_grad()')
print('=' * 50)
torch.manual_seed(42)
demo_model = nn.Linear(4, 1)
demo_opt = optim.SGD(demo_model.parameters(), lr=0.01)
x_demo = torch.randn(8, 4)
y_demo = torch.randn(8, 1)

print('\nWithout zero_grad (WRONG — gradients accumulate):')
for i in range(3):
    loss = nn.MSELoss()(demo_model(x_demo), y_demo)
    loss.backward()
    print(f'  Step {i}: grad norm = {demo_model.weight.grad.norm():.4f}')

print('\nWith zero_grad (CORRECT):')
for i in range(3):
    demo_opt.zero_grad()
    loss = nn.MSELoss()(demo_model(x_demo), y_demo)
    loss.backward()
    print(f'  Step {i}: grad norm = {demo_model.weight.grad.norm():.4f}')

# Other common mistakes
print('\n' + '=' * 50)
print('OTHER COMMON MISTAKES:')
print('=' * 50)
print('#2: Using CrossEntropy for regression (use MSELoss)')
print('#3: Model on GPU, data on CPU (use .to(device) for both)')
print('#4: Forgetting model.eval() during inference')
print('#5: Not normalizing input features')

# Best practices
print('\n' + '=' * 50)
print('BEST PRACTICES:')
print('=' * 50)
checklist = [
    ('Normalize your data', 'StandardScaler or (x - mean) / std'),
    ('Use Adam with lr=0.001', 'Best default for most problems'),
    ('Print shapes everywhere', 'Catch dimension bugs early'),
    ('Track test/val loss', 'Detect overfitting'),
    ('Use model.eval()', 'Consistent inference predictions'),
    ('Set random seeds', 'torch.manual_seed(42) for reproducibility'),
]
for practice, why in checklist:
    print(f'  [{practice}] — {why}')

> **For your report:** Explain why forgetting `zero_grad()` causes gradient accumulation. What would happen if you used `CrossEntropyLoss` for a regression task? Why is `model.eval()` important during inference?

---

# Part 13: Save & Conclusion

## 13.1 Save the Trained Model

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SAVE AND LOAD MODEL                                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Save
save_path = 'pytorch_fundamentals_model.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': {'input_dim': 8, 'hidden1': 64, 'hidden2': 32},
    'metrics': {'mse': mse, 'rmse': rmse, 'mae': mae, 'r2': r2},
}, save_path)
print(f'Model saved to: {save_path}')
print(f'R\u00b2 = {r2:.4f}')

# Load (demonstrate)
checkpoint = torch.load(save_path, weights_only=True)
loaded_model = HousingNet(**checkpoint['architecture'])
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model.eval()

# Verify
with torch.no_grad():
    loaded_pred = loaded_model(X_test[:5])
    original_pred = model.cpu()(X_test[:5])
print(f'\nLoaded model matches original: {torch.allclose(loaded_pred, original_pred)}')

## 13.2 Summary of All Saved Figures

Every figure generated by this notebook is saved as a PNG for your report:

| Figure File | Content | Report Section |
|---|---|---|
| `asgn_fig_autograd_functions.png` | 4 functions with their autograd derivatives | Autograd |
| `asgn_fig_feature_distributions.png` | Histogram of all 9 variables | Dataset |
| `asgn_fig_correlations.png` | Scatter plots: feature vs target with correlation | Dataset |
| `asgn_fig_learning_rate_demo.png` | Gradient descent with 4 learning rates | Optimizers |
| `asgn_fig_training_curves.png` | Training and test loss over epochs | Training |
| `asgn_fig_evaluation.png` | Predicted vs actual scatter + residual histogram | Results |
| `asgn_fig_exp_architecture.png` | Experiment A: architecture comparison | Experiments |
| `asgn_fig_exp_learning_rate.png` | Experiment B: learning rate comparison | Experiments |
| `asgn_fig_exp_optimizer.png` | Experiment C: optimizer comparison | Experiments |

## 13.3 Report Writing Guide

### What to Write in Each Section

**1. Introduction (1 page)**
- What is PyTorch? How does it differ from NumPy?
- Explain tensors, autograd, and the computational graph
- Describe `nn.Module`, loss functions, and optimizers
- Overview of the 4-line training pattern

**2. Dataset (0.5 page)**
- Describe California Housing: 20,640 samples, 8 features, target
- Include `asgn_fig_feature_distributions.png` and `asgn_fig_correlations.png`
- Explain preprocessing: normalization (why needed?) and train/test split

**3. Autograd (0.5 page)**
- Explain `requires_grad`, `.backward()`, and `.grad`
- Include `asgn_fig_autograd_functions.png`
- Discuss `no_grad()` and `detach()` — when to use each

**4. Model Architecture (0.5 page)**
- Network architecture: [8, 64, 32, 1] with ReLU
- Parameter count breakdown by layer
- `nn.Sequential` vs custom `nn.Module`

**5. Training (1 page)**
- The 4-line training pattern (explain each step)
- Include `asgn_fig_training_curves.png`
- Discuss convergence and train/test gap
- Include `asgn_fig_learning_rate_demo.png`

**6. Results (0.5 page)**
- Include all metrics: MSE, RMSE, MAE, R\u00b2
- Include `asgn_fig_evaluation.png`
- Discuss best/worst predictions

**7. Experiments (1 page)**
- For EACH experiment: include the comparison plot and results table
- State your observation and explain WHY
- Recommend the best setting for each hyperparameter

**8. Conclusion (0.5 page)**
- Summary of key PyTorch concepts learned
- What you learned about training neural networks with PyTorch
- Limitations and potential improvements

---

<center>

### Assignment Complete

**Remember to:**
1. Run all cells from top to bottom before submission
2. Ensure all outputs and figures are visible
3. Write your report using the generated figures
4. Save your model file

---

*Apeiron AI | "Boundless Possibilities, Infinite Potential"*
*© 2026 | www.aperionaiml.com*

</center>